In [1]:
import sys
import os
import time
from lightkurve import search_targetpixelfile
from lightkurve import TessTargetPixelFile
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from keras.models import load_model
from keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Conv1D, MaxPooling1D, Flatten
from wotan import slide_clip
from wotan import transit_mask, flatten
from astropy.stats import sigma_clip
from astropy import units as u
import csv
import shutil
from scipy.interpolate import interp1d
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import make_forecasting_frame
from tsfresh.utilities.dataframe_functions import impute
from statsmodels.tsa.seasonal import seasonal_decompose
from multiprocessing import Pool
import multiprocessing
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.signal import find_peaks
from astropy.timeseries import BoxLeastSquares
from scipy.interpolate import interp1d

In [2]:
%matplotlib inline

In [3]:
np.random.seed(42)
tf.random.set_seed(42)

In [4]:
value_df = 1000
cwd = os.getcwd()
dirname = cwd[len(cwd)-len("Satellite Datasets"):len(cwd)]
if(dirname != 'Satellite Datasets'):
    os.chdir('./Satellite Datasets')
star_check = pd.read_csv("exoplanet_star_trend_flux.csv")
star_check = star_check.drop(['tid'],axis=1)
star_check_y = star_check[['confirmed_planet']]
star_check = star_check.reset_index().drop('index',axis=1)
star_check = star_check.drop(['confirmed_planet'],axis=1).apply(lambda row: (row - row.min()) / (row.max() - row.min()), axis=1)
star_check[['confirmed_planet']]=star_check_y

In [8]:
star_check = star_check.sample(frac = 1)
star_check = star_check.dropna() #
X_train = pd.concat([star_check[star_check['confirmed_planet']==0][0:3500],star_check[star_check['confirmed_planet']==1][0:3500]])
X_test = pd.concat([star_check[star_check['confirmed_planet']==0][3500:3600],star_check[star_check['confirmed_planet']==1][3500:3600]])
# Increase the testing data size. using kfold cross validation. use 10% of the data for testing
y_train = X_train[['confirmed_planet']]
y_test = X_test[['confirmed_planet']]
X_train = X_train.drop('confirmed_planet',axis=1)
X_test = X_test.drop('confirmed_planet',axis=1)

In [9]:
for i in range(10):
    np.random.seed(i)
    tf.random.set_seed(i)
    model = Sequential([
        Conv1D(filters=16, kernel_size=3, activation='relu', input_shape=(1000,1)),
        MaxPooling1D(),
        Conv1D(filters=32, kernel_size=3, activation='relu'),
        MaxPooling1D(),
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        MaxPooling1D(),
        Conv1D(filters=32, kernel_size=3, activation='relu'),
        Conv1D(filters=32, kernel_size=3, activation='relu'),
        MaxPooling1D(),
        Conv1D(filters=16, kernel_size=5, activation='relu'),
        MaxPooling1D(),
        Dropout(0.5),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train,y_train,epochs=10,batch_size=128,validation_data=(X_test,y_test))
    print(model.evaluate(X_test,y_test))
    

Epoch 1/10
55/55 [==============================] - 3s 36ms/step - loss: 0.6899 - accuracy: 0.5260 - val_loss: 0.6765 - val_accuracy: 0.5700
Epoch 2/10
55/55 [==============================] - 2s 34ms/step - loss: 0.6742 - accuracy: 0.5856 - val_loss: 0.6580 - val_accuracy: 0.5950
Epoch 3/10
55/55 [==============================] - 2s 33ms/step - loss: 0.6534 - accuracy: 0.6174 - val_loss: 0.6385 - val_accuracy: 0.6450
Epoch 4/10
55/55 [==============================] - 2s 33ms/step - loss: 0.6471 - accuracy: 0.6273 - val_loss: 0.6459 - val_accuracy: 0.6550
Epoch 5/10
55/55 [==============================] - 2s 33ms/step - loss: 0.6383 - accuracy: 0.6384 - val_loss: 0.6715 - val_accuracy: 0.5900
Epoch 6/10
55/55 [==============================] - 2s 33ms/step - loss: 0.6382 - accuracy: 0.6403 - val_loss: 0.6327 - val_accuracy: 0.6400
Epoch 7/10
55/55 [==============================] - 2s 33ms/step - loss: 0.6281 - accuracy: 0.6504 - val_loss: 0.6287 - val_accuracy: 0.6400
Epoch 8/10
55

In [ ]:
model.load_weights('model_v2.hdf5')

In [ ]:
model.evaluate(X_test,y_test)

7/7 [==============================] - 0s 4ms/step - loss: 0.2638 - accuracy: 0.9250


[0.26383036375045776, 0.925000011920929]